# 图像的读取与显示

本 Notebook 系统讲解 OpenCV 中图像的读取、显示、通道顺序、窗口机制与常见问题。

主要内容：
1. 图像读取 `cv2.imread()`
2. `flags` 的本质：解码阶段的处理
3. 查看原始数据与解码数据
4. 原始图像文件的整体结构分层
5. 灰度图像与彩色图像的头部差异、灰色深度、灰度文件实战
6. 图像显示 `cv2.imshow()`、`cv2.waitKey()`、`cv2.destroyAllWindows()`
7. OpenCV 的 BGR 通道顺序与 Matplotlib 的 RGB 差异
8. 灰度图读取与显示
9. 封装显示函数与常见错误
10. 附录：透明度、alpha、显示背景的深入说明

# 1. 图像读取

## 1.1 `cv2.imread()` 基本用法

```python
img = cv2.imread(filename, flags)
```

参数说明：
- `filename`：图像路径，字符串类型。cv2.imread() 默认支持 JPEG、PNG、BMP、TIFF、GIF（仅第一帧）等常见格式；WebP 是否支持取决于 OpenCV 的编译配置，若读取失败可优先怀疑这一点。完整格式说明见 附录 D。
- `flags`：读取模式，常用值如下：
  - `cv2.IMREAD_COLOR`：默认值，读取彩色图像，忽略透明度，返回 3 通道 BGR 图像。
  - `cv2.IMREAD_GRAYSCALE`：读取为灰度图像，返回单通道图像。
  - `cv2.IMREAD_UNCHANGED`：读取原图，包括 alpha 通道。

返回值：
- 成功时返回 `numpy.ndarray`，数据类型通常为 `uint8`，取值范围 0~255。
- 失败时返回 `None`，通常是因为路径错误、文件不存在或格式不支持。

**重要理解**

原始图像文件里的数据是固定的，`flags` 并不会改变文件本身。
它决定的是 **读取解码之后，返回给你的 numpy 数组是什么形态**：
几个通道、什么颜色空间、是否保留 alpha。

换句话说，`flags` 是一种 **解码阶段的处理**，而不是对已经读进来的数组做二次操作。

① 图像读取入口参数：
  - cv2.IMREAD_COLOR：彩色图像
  - cv2.IMREAD_GRAYSCALE：灰度图像
  - cv2.IMREAD_UNCHANGED：保留 alpha 通道

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作


In [ ]:
img = cv2.imread('01_Picture/T-shirt.png')              # 读取图像，默认 IMREAD_COLOR，返回 3 通道 BGR
print(type(img))                                        # 打印类型，成功为 numpy.ndarray，失败为 NoneType
print(img.shape if img is not None else '读取失败')     # 成功打印 (高, 宽, 通道数)，失败打印提示
img                                                     # Notebook 直接输出，查看数组内容与 dtype

## 1.2 `flags` 的本质：解码阶段的处理

前面已经强调：**原始文件数据固定，`flags` 决定读进来之后怎么解释和转换这些数据。**
这一节把这个过程展开讲清楚。

### 1.2.1 图像文件里存的是什么

一张 JPG/PNG 文件，本质上是 **压缩编码后的像素数据**，比如：
- 彩色 JPG 里存的是 YCbCr 之类的编码信息
- PNG 可能存了 RGB + alpha

它并不是直接按 BGR 或灰度排列的原始数组。

### 1.2.1.1 直观示例：同一份文件，三种模式读出来什么样

光看文字不够直观。这里用同一份文件 `T-shirt.png`，分别按三种模式读进来，
对比 `shape`、`dtype`、像素值，并把图像并排画出来。

**这张 `T-shirt.png` 是带 alpha 的 PNG**，所以：

- `IMREAD_COLOR` 会丢弃 alpha，返回 3 通道 BGR；
- `IMREAD_GRAYSCALE` 返回单通道灰度；
- `IMREAD_UNCHANGED` 会保留 alpha，返回 4 通道 BGRA。

观察重点：
- 三种读法的 `shape`、`dtype`、像素值；
- **`IMREAD_UNCHANGED` 会多出第 4 个通道**，这就是 alpha；
- `IMREAD_COLOR` 的前 3 通道等于 `IMREAD_UNCHANGED` 的前 3 通道；
- 用 Matplotlib 显示时，彩色需要 `BGR -> RGB`，否则颜色会偏；
- **`plt.imshow` 对 4 通道数组不会自动按 alpha 做合成**，所以直接显示时，`IMREAD_UNCHANGED` 看起来和 `IMREAD_COLOR` 一模一样。要让 alpha 真正起作用，需要**手动合成**，详见 1.2.1.2。

In [ ]:
path = '01_Picture/T-shirt.png'                                # 定义图像路径，供三种读取模式共用

img_color     = cv2.imread(path, cv2.IMREAD_COLOR)             # 彩色模式：返回 3 通道 BGR，丢弃 alpha
img_gray      = cv2.imread(path, cv2.IMREAD_GRAYSCALE)         # 灰度模式：返回单通道灰度
img_unchanged = cv2.imread(path, cv2.IMREAD_UNCHANGED)         # 原样模式：保留 alpha，返回 4 通道 BGRA

print('COLOR     shape:', img_color.shape,     'dtype:', img_color.dtype)      # 打印彩色图形状与类型
print('GRAYSCALE shape:', img_gray.shape,      'dtype:', img_gray.dtype)       # 打印灰度图形状与类型
print('UNCHANGED shape:', img_unchanged.shape, 'dtype:', img_unchanged.dtype)  # 打印原样图形状与类型

In [ ]:
img_color_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)          # BGR 转 RGB，便于 Matplotlib 正确显示

plt.figure(figsize=(12, 4))                                         # 创建画布，宽 12 英寸、高 4 英寸

plt.subplot(1, 3, 1)                                                # 创建 1 行 3 列子图第 1 个
plt.imshow(img_color_rgb)                                           # 显示已转 RGB 的彩色图
plt.title('IMREAD_COLOR\nBGR -> RGB, shape={}'.format(img_color.shape))  # 设置子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.subplot(1, 3, 2)                                                # 创建 1 行 3 列子图第 2 个
plt.imshow(img_gray, cmap='gray')                                   # 以灰度色图显示单通道灰度图
plt.title('IMREAD_GRAYSCALE\nshape={}'.format(img_gray.shape))      # 设置子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.subplot(1, 3, 3)                                                # 创建 1 行 3 列子图第 3 个
plt.imshow(cv2.cvtColor(img_unchanged, cv2.COLOR_BGRA2RGB))         # BGRA 转 RGB 显示，alpha 被 imshow 忽略
plt.title('IMREAD_UNCHANGED\nshape={}'.format(img_unchanged.shape)) # 设置子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.tight_layout()                                                  # 自动调整子图间距
plt.show()                                                          # 渲染并显示图像

从上面三张图可以看到：
- 左图和中图内容一致，但左图是 3 通道彩色，中图是 1 通道灰度；
- 右图比左图多出第 4 个通道，因为 `T-shirt.png` 是带 alpha 的 PNG，`IMREAD_UNCHANGED` 会保留 alpha；
- 左图（3 通道）和右图（4 通道）的前 3 个通道一致，说明 `IMREAD_COLOR` 只是丢弃了 alpha，没有改动颜色；
- **三张图看起来一样，但不代表它们数据相同**：左图没有 alpha，右图有 alpha，两者在数据层面不同。

### 1.2.1.2 军绿色背景从何而来？alpha 到底怎么才能“显形”？

前面展示三张子图时，细心的读者会注意到一圈“军绿色背景”。
这一节把这个现象讲清楚，并给出让 alpha 真正起作用的实验。

#### 1.2.1.2.1 军绿色不是 figure 底色，也不是“背景色”

对 `T-shirt.png` 左上角 `(0,0)` 采样，看它的 BGRA：

In [ ]:
img4 = cv2.imread('01_Picture/T-shirt.png', cv2.IMREAD_UNCHANGED)   # 以原样模式读取，得到 4 通道 BGRA
print('左上角 BGRA:', img4[0, 0])                                   # 打印左上角像素的 BGRA 值

典型输出：

```
左上角 BGRA: [ 76 112  71   0]
```

解读：

- **B=76, G=112, R=71** —— 军绿色；
- **A=0** —— 完全透明。

结论：

> **军绿色是文件里透明区域保存的 BGR 值，不是 figure 底色，也不是图像真实背景。**

统计整张图的 alpha 分布：

In [ ]:
img4 = cv2.imread('01_Picture/T-shirt.png', cv2.IMREAD_UNCHANGED)   # 读取 4 通道 BGRA 图像
alpha = img4[:, :, 3]                                               # 取出 alpha 通道（第 4 个通道）

print('alpha 最小值/最大值:', alpha.min(), alpha.max())             # 打印 alpha 最小值与最大值
print('alpha 等于 255 的像素占比:', (alpha == 255).mean())          # 统计完全不透明像素比例
print('alpha 等于 0 的像素占比:  ', (alpha == 0).mean())            # 统计完全透明像素比例
print('alpha 介于 1~254 的占比:  ', ((alpha > 0) & (alpha < 255)).mean())  # 统计半透明像素比例

典型输出：

```
alpha 最小值/最大值: 0 255
alpha 等于 255 的像素占比: 0.46581875   ≈ 46.6%
alpha 等于 0 的像素占比:   0.52566875   ≈ 52.6%
alpha 介于 1~254 的占比:   0.0085125    ≈ 0.85%
```

说明这张图确实有大量透明区域。

#### 1.2.1.2.2 为什么直接 imshow 看不出差异

对 4 通道数组，`plt.imshow` 的默认行为更接近“**把前 3 通道当 RGB、忽略 alpha**”。
所以：

- 左图（3 通道）：直接画 BGR，看到军绿色；
- 中图（灰度）：军绿按亮度公式转成中灰；
- 右图（4 通道）：`imshow` 忽略 alpha，仍画 BGR，看到军绿色。

**“看到军绿色”不代表“背景是军绿色”，而是“透明区域的 BGR 被错误地直接画了出来”。**

#### 1.2.1.2.3 让 alpha 显形：同一张图，合成到不同底色

要让 alpha 真正起作用，必须**手动合成**，把公式跑一遍：

```
结果 = 前景 × alpha + 背景 × (1 - alpha)
```

下面这段代码，把同一张 PNG 分别合成到**黑底**和**白底**上。

In [ ]:
path = '01_Picture/T-shirt.png'                                     # 图像路径
img4 = cv2.imread(path, cv2.IMREAD_UNCHANGED)                       # 读取 4 通道 BGRA 图像
b, g, r, a = cv2.split(img4)                                        # 将 4 个通道拆分为 B、G、R、A
a = a.astype(np.float32) / 255.0                                    # 将 alpha 归一化到 0~1 的浮点数
a = a[..., None]                                                    # 将 alpha 扩展为 (H, W, 1)，便于广播

fg = img4[:, :, :3].astype(np.float32)                              # 取出前景 BGR 三通道并转 float32
bg_black = np.zeros_like(fg)                                        # 构造黑色背景，形状与前景一致
bg_white = np.full_like(fg, 255, dtype=np.float32)                  # 构造白色背景，形状与前景一致

out_black = (fg * a + bg_black * (1 - a)).astype(np.uint8)          # 按 alpha 合成到黑底，转回 uint8
out_white = (fg * a + bg_white * (1 - a)).astype(np.uint8)          # 按 alpha 合成到白底，转回 uint8

plt.figure(figsize=(12, 4))                                         # 创建画布

plt.subplot(1, 3, 1)                                                # 第 1 个子图
plt.imshow(cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB))  # 彩色读取并转 RGB 显示
plt.title('IMREAD_COLOR (alpha ignored)')                           # 子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.subplot(1, 3, 2)                                                # 第 2 个子图
plt.imshow(cv2.cvtColor(out_black, cv2.COLOR_BGR2RGB))              # 显示合成到黑底的结果
plt.title('UNCHANGED + black background')                           # 子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.subplot(1, 3, 3)                                                # 第 3 个子图
plt.imshow(cv2.cvtColor(out_white, cv2.COLOR_BGR2RGB))              # 显示合成到白底的结果
plt.title('UNCHANGED + white background')                           # 子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.tight_layout()                                                  # 自动调整布局
plt.show()                                                          # 显示图像

预期结果：

- **左图**：军绿色背景（`imshow` 忽略 alpha）；
- **中图**：背景变**黑**（alpha=0 → 结果=黑）；
- **右图**：背景变**白**（alpha=0 → 结果=白）。

**同一份文件，三种背景，结果不同**——这就是 alpha 起作用的最直接证据。

#### 1.2.1.2.4 让 alpha 显形的其它办法

**办法一：单独画 alpha 通道**

In [ ]:
path = '01_Picture/T-shirt.png'                                     # 图像路径
img4 = cv2.imread(path, cv2.IMREAD_UNCHANGED)                       # 读取 4 通道 BGRA 图像
alpha = img4[:, :, 3]                                               # 取出 alpha 通道

plt.figure(figsize=(8, 4))                                          # 创建画布

plt.subplot(1, 2, 1)                                                # 第 1 个子图
plt.imshow(cv2.cvtColor(img4, cv2.COLOR_BGRA2RGB))                  # BGRA 转 RGB 后显示前 3 通道
plt.title('UNCHANGED, first 3 channels')                            # 子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.subplot(1, 2, 2)                                                # 第 2 个子图
plt.imshow(alpha, cmap='gray')                                      # 以灰度色图显示 alpha 通道
plt.title('alpha channel (white=opaque, black=transparent)')        # 子图标题
plt.axis('off')                                                     # 关闭坐标轴

plt.tight_layout()                                                  # 自动调整布局
plt.show()                                                          # 显示图像

**办法二：人为把某块 alpha 置 0，再合成**

In [ ]:
img4 = cv2.imread('01_Picture/T-shirt.png', cv2.IMREAD_UNCHANGED).copy()  # 读取并复制一份 4 通道图像
img4[0:100, 0:100, 3] = 0                                           # 将左上角 100x100 区域的 alpha 置 0

b, g, r, a = cv2.split(img4)                                        # 拆分通道为 B、G、R、A
a = a.astype(np.float32) / 255.0                                    # 归一化 alpha 到 0~1
a = a[..., None]                                                    # 扩展 alpha 维度为 (H, W, 1)
fg = img4[:, :, :3].astype(np.float32)                              # 取出前景 BGR 并转 float32
bg = np.full_like(fg, 255, dtype=np.float32)                        # 构造白色背景
out = (fg * a + bg * (1 - a)).astype(np.uint8)                      # 按 alpha 合成并转回 uint8

plt.figure(figsize=(6, 6))                                          # 创建画布
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))                    # 显示合成结果（转 RGB）
plt.title('Top-left 100x100 set to alpha=0, composited on white')   # 标题
plt.axis('off')                                                     # 关闭坐标轴
plt.show()                                                          # 显示图像

**办法三：用 imshow 的 alpha 参数，传一个渐变 alpha**

In [ ]:
img3 = cv2.imread('01_Picture/T-shirt.png', cv2.IMREAD_COLOR)       # 以彩色模式读取 3 通道 BGR
img3_rgb = cv2.cvtColor(img3, cv2.COLOR_BGR2RGB)                    # 转为 RGB，便于 Matplotlib 显示

h, w = img3_rgb.shape[:2]                                           # 获取图像高度与宽度
alpha = np.tile(np.linspace(0, 1, w), (h, 1))                       # 构造从左到右 0~1 的 alpha 渐变矩阵

plt.figure(figsize=(6, 6))                                          # 创建画布
plt.imshow(img3_rgb, alpha=alpha)                                   # 使用渐变 alpha 显示图像
plt.title('alpha gradient from 0 to 1 (left to right)')             # 标题
plt.axis('off')                                                     # 关闭坐标轴
plt.show()                                                          # 显示图像

#### 1.2.1.2.5 小结

- **军绿色从何而来**：来自 PNG 文件里透明区域保存的 BGR 值。
  它不是 figure 底色，也不是“背景色”，只是“被标记为透明的像素仍然携带的颜色”。
- **alpha=0 与没有 alpha 的区别**：
  - **没有 alpha**：只有 BGR，结构上不存在“透明”概念；
  - **alpha=0**：有 BGR+A，明确携带“完全透明”的信息。
  两者显示可能一样，但数据能力不同。
- **alpha 显形的条件**：必须**参与一次合成**，或作为显式 alpha 参数传给显示函数。
  直接 `imshow` 4 通道数组，通常等于忽略 alpha。
- **最直观的实验**：同一张 PNG，分别合成到黑底与白底，透明区域一个变黑一个变白。

### 1.2.2 `cv2.imread()` 做的事

读取时，OpenCV 会：
1. 解码文件 → 还原出像素信息
2. **根据 `flags` 决定输出成什么形式**

所以 `flags` 不是“改变原文件”，而是 **改变解码后返回给你的 numpy 数组的形态**。

### 1.2.3 三种模式到底做了什么

**`IMREAD_COLOR`**
- 把图像解码成 3 通道
- 通道顺序排成 **BGR**
- 如果有 alpha 通道，**丢弃**

这里其实发生了两件事：
- 通道重排成 BGR（这是 OpenCV 的历史约定）
- alpha 被扔掉

**`IMREAD_GRAYSCALE`**
- 把彩色信息 **按加权公式转成亮度**
- 常见公式近似：`Y = 0.299R + 0.587G + 0.114B`
- 输出单通道

所以灰度图不是文件里本来就有的一份灰度数据，而是 **读取时现场算出来的**。

**`IMREAD_UNCHANGED`**
- 文件里有什么就还原什么
- 有 alpha 就保留，变成 4 通道 BGRA
- 不做通道丢弃、不做灰度转换

### 1.2.4 一个直观对比

假设原图是一个带透明背景的 PNG：

- 使用 `IMREAD_COLOR` 时，返回 3 通道 BGR，透明区域可能变黑或变白。
- 使用 `IMREAD_GRAYSCALE` 时，返回单通道灰度，透明信息丢失。
- 使用 `IMREAD_UNCHANGED` 时，返回 4 通道 BGRA，透明信息保留。

同一份文件，三种读法得到三个不同的数组。

### 1.2.5 与 `cvtColor` 的区别

如果你先 `IMREAD_UNCHANGED` 读进来，再自己用 `cv2.cvtColor` 转灰度，也能得到类似结果。
区别只是：
- `flags` 是在 **解码时一步到位**
- `cvtColor` 是对 **已解码的数组再做一次转换**

### 1.2.6 常量对应的整数值

这些常量对应的实际整数值是：
- `cv2.IMREAD_COLOR` = 1
- `cv2.IMREAD_GRAYSCALE` = 0
- `cv2.IMREAD_UNCHANGED` = -1

所以你也可以直接写 `cv2.imread('cat.jpg', 0)` 来读灰度图，效果一样，但 **建议用常量名**，可读性更好。

一句话总结：`flags` 就是告诉 OpenCV “我要彩色、灰度，还是连透明通道都要”。

## 1.3 查看原始数据与解码数据

前面讲了 `flags` 决定解码输出形态，这一节从 **字节层到数组层** 把这件事看完整。
我们会依次查看：

1. 原始图像文件的整体结构分层
2. 文件头部数据（magic number、格式标识、元信息）
3. 原始字节流（解码前的二进制数据）
4. 解码后的像素数据（numpy 数组）
5. 不同 `flags` 下的数据差异
6. 灰度图像与彩色图像在头部上的差异，以及灰色深度（位深）

### 1.3.0 原始图像文件的整体结构

从“文件字节”这个层面看，一张原始图像文件通常由下面几大部分组成。
不同格式（JPEG、PNG、BMP、TIFF 等）具体字段不同，但 **结构分层是共通的**。

整体可以想象成一个从上到下排列的六层结构：

1. **文件签名 / Magic Number**：标识格式
2. **元信息 / 头部段**：版本、密度、色彩配置、EXIF…
3. **图像描述段**：宽、高、位深、通道数、压缩方式
4. **辅助表**：量化表、霍夫曼表、调色板…
5. **压缩像素数据**：真正的图像内容（编码后）
6. **结束标记**：文件结束

下面逐层展开。

#### 第一层：文件签名（Magic Number）

文件最开头几个字节，用来告诉解码器“我是什么格式”。

常见格式的开头字节如下：

- JPEG 以 `ff d8` 开头；
- PNG 以 `89 50 4e 47 0d 0a 1a 0a` 开头；
- GIF 以 `47 49 46 38` 开头，也就是 `GIF8`；
- BMP 以 `42 4d` 开头，也就是 `BM`；
- TIFF 以 `49 49 2a 00` 或 `4d 4d 00 2a` 开头；
- WebP 以 `52 49 46 46` 开头，中间隔若干字节后再出现 `57 45 42 50`。

它的作用是：让程序不用看扩展名也能判断格式。

#### 第二层：元信息 / 头部段

在签名之后，通常是一些 **描述文件本身** 的信息，包括：

- 格式版本，比如 JFIF 1.1、PNG 1.2；
- 密度（DPI），也就是打印时的物理分辨率；
- 像素宽高比；
- 缩略图（可选）；
- EXIF，比如相机型号、拍摄时间、GPS、方向、曝光参数；
- ICC 色彩配置文件，描述颜色空间；
- 注释、版权信息等。

这一部分 **不包含像素本身**，但会影响图像怎么被解释和显示。

#### 第三层：图像描述段

这一层才真正告诉解码器“图像长什么样”，通常包括：

- 宽度、高度，也就是图像像素尺寸；
- 位深（精度），即每个样本用多少 bit，例如 8、12、16；
- 通道数 / 分量数，1 表示灰度，3 表示彩色，4 表示带 alpha；
- 色彩空间，可能是灰度、RGB、YCbCr、CMYK 或调色板；
- 压缩方式，可能是无压缩、DCT、LZW、Deflate 等；
- 扫描方式，比如基线、渐进、隔行。

在 JPEG 里，这些信息集中在 **SOF 段**；在 PNG 里，在 **IHDR 块**；在 BMP 里，在 **DIB Header**。

##### 【延伸一】宽高的单位是像素

宽高的单位是 **像素（pixel）**，不是厘米、英寸这类物理单位。

- 宽表示水平方向有多少个像素点，高表示垂直方向有多少个像素点。
- 例如 `512 × 512`，意思是水平 512 个像素、垂直 512 个像素。
- 在 numpy 数组里，形状是 `(height, width, channels)`，注意 **高在前、宽在后**。
- 想要换算成物理尺寸，需要结合 **DPI / 密度**（在第 2 层元信息的 APP0 段里）。
- 例如 512×512、DPI=120，打印物理尺寸约为 `512 / 120 ≈ 4.27 英寸`。
- 所以 **SOF 段里的宽高是像素数**，**APP0 段里的 DPI 是每英寸像素数**，两者是不同层的信息，不要混在一起。

##### 【延伸二】像素没有物理大小，显示大小由什么决定

这是理解数字图像的一个关键点。像素本身 **没有固定物理大小**，也 **没有“像素之间的距离”这个概念**。

**一、像素是采样点，不是方块**

很多人把像素想象成“一个个小方块拼成图”，这其实是一种 **可视化比喻**，不是本质。

- 图像是对连续场景做 **离散采样** 的结果；
- 每个像素是某个位置上的 **一个采样值**（灰度或颜色）；
- 像素在数学上更接近“一个点”，而不是“一个有面积的小方块”。

所谓“像素之间”，其实指的是 **相邻采样点之间的间隔**，而这个间隔是 **采样时决定的**，不是图像本身携带的固定量。

**二、像素有“大小”吗？**

分两个层面看：

在数字层面，没有物理大小：

- 一个像素就是一个数值，比如 `uint8` 的 0~255；
- 它不携带“我占多少毫米”的信息；
- 两个像素在数组里相邻，只是索引差 1，没有物理距离。

在显示/打印层面，才有物理大小：

- 屏幕由 **PPI（每英寸像素数）** 决定，比如 96 PPI 屏幕上 1 个像素 ≈ 1/96 英寸；
- 打印由图像的 **DPI** 决定，比如 300 DPI 时 1 个像素 ≈ 1/300 英寸。

同一张 512×512 的图：

- 在 96 PPI 屏幕上，物理尺寸约 `512/96 ≈ 5.33 英寸`；
- 在 300 DPI 打印时，物理尺寸约 `512/300 ≈ 1.71 英寸`。

**像素数没变，只是物理解释变了。**

**三、图像显示出来的大小是怎么控制的？**

图像文件里根本没有“显示大小”，它只记录两类信息：

- **像素矩阵**：宽多少个像素、高多少个像素；
- **元信息（可选）**：DPI / 密度、像素宽高比。

它 **不记录** “应该显示成多少厘米、多少英寸”。

真正决定“看起来多大”的，是下面三者：

- **图像本身的像素数**，比如 512×512，这是显示时的“原料”；
- **显示设备的像素密度（PPI）**，屏幕每英寸有多少个物理像素，比如手机 300~500 PPI，显示器 90~220 PPI，老屏幕 72~96 PPI；
- **软件怎么映射**，显示软件（浏览器、看图器、OpenCV 窗口）决定是 1 个图像像素对 1 个屏幕物理像素（100% 显示），还是缩放后再显示。

这三者一组合，才是你看到的“大小”。

**四、常见几种情形**

100% 显示（1 图像像素 = 1 屏幕像素）：

- 显示尺寸 = 像素数 / 屏幕 PPI；
- 例如 512×512 在 96 PPI 屏幕上约 `512/96 ≈ 5.33 英寸`。

缩放显示：

- 浏览器、看图器为了适配窗口，会重新采样；
- 放大时 1 个原像素 → 多个屏幕像素；缩小时多个原像素 → 1 个屏幕像素；
- 这时“显示大小”由窗口和缩放比例决定，和 PPI 的关系变成间接的。

打印：

- 打印时由图像的 DPI 决定；
- DPI=300、512×512 → 打印尺寸 `512/300 ≈ 1.71 英寸`；
- 如果你在打印软件里改 DPI，同一张图的打印尺寸就变了，但像素数没变。

**五、DPI 到底在这里起什么作用**

DPI 是 **元信息**，它的作用只有一个：告诉打印或排版软件“我建议每个像素占多少物理尺寸”。

- 如果软件 **尊重** DPI → 打印尺寸 = 像素数 / DPI；
- 如果软件 **忽略** DPI（很多看图器、浏览器就是这样）→ 按 100% 或按窗口缩放，DPI 不生效；
- 屏幕显示通常不看 DPI，而是看屏幕自己的 PPI 和软件缩放。

所以：

- **屏幕显示大小** ≈ 像素数 ÷ 屏幕 PPI × 缩放比例；
- **打印大小** ≈ 像素数 ÷ 图像 DPI。

**六、用 OpenCV / Matplotlib 验证**

下面用四个逐步进行的实验来验证“显示大小到底由什么决定”。
每个实验后面紧跟对应的可运行代码，方便边看说明边运行。

##### 实验一：subplots 并排显示

- 512×512 和 256×256 会 **看起来一样大**；
- 因为 `subplots` 把每个子图缩放到相同格子尺寸；
- 说明 **Matplotlib 里，显示大小由 figure 布局决定，不是由像素数决定**。

In [ ]:
import cv2                                                    # 导入 OpenCV，用于绘制圆、写字、缩放
import matplotlib.pyplot as plt                               # 导入 Matplotlib pyplot，用于子图布局
import numpy as np                                            # 导入 NumPy，用于创建数组

h, w = 512, 512                                               # 定义图像高度与宽度
img = np.zeros((h, w, 3), dtype=np.uint8)                     # 创建 (512,512,3) 全零 uint8 图像，初始黑色
img[0:256, 0:256]     = [255, 0, 0]                           # 左上角填蓝色（BGR 下 [255,0,0] 为蓝）
img[0:256, 256:512]   = [0, 255, 0]                           # 右上角填绿色
img[256:512, 0:256]   = [0, 0, 255]                           # 左下角填红色
img[256:512, 256:512] = [255, 255, 0]                         # 右下角填青色
cv2.circle(img, (256, 256), 120, (255, 255, 255), -1)         # 在中心画半径 120 的白色实心圆
cv2.putText(img, '512x512', (150, 270),                       # 在图上写文字 '512x512'
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3)      # 字体、缩放 1.5、黑色、线宽 3

small = cv2.resize(img, (w // 2, h // 2), interpolation=cv2.INTER_AREA)  # 缩小为 256x256，使用区域插值
print('原始 shape:', img.shape)                               # 打印原图 shape (512,512,3)
print('缩小 shape:', small.shape)                             # 打印缩小图 shape (256,256,3)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))               # 创建 1 行 2 列子图，画布 12x6
axes[0].imshow(img)                                           # 左侧显示原图
axes[0].set_title(f'Original  {w}x{h}')                       # 左侧标题
axes[0].axis('off')                                           # 关闭左侧坐标轴
axes[1].imshow(small)                                         # 右侧显示缩小图
axes[1].set_title(f'Half  {w//2}x{h//2}')                     # 右侧标题
axes[1].axis('off')                                           # 关闭右侧坐标轴
plt.tight_layout()                                            # 自动调整间距
plt.show()                                                    # 显示图像

print('现象：左右一样大。')                                  # 现象说明
print('原因：subplots 强制对齐格子，图像被缩放去填满格子。')  # 原因说明

##### 实验二：GridSpec 让左右子图宽度不同

- 让左边占 2 列、右边占 1 列；
- 这时左边明显比右边大；
- 说明 **只有让子图本身占据不同的物理宽度，才能看出“像素数不同 → 显示大小不同”**。

In [ ]:
from matplotlib.gridspec import GridSpec                          # 导入 GridSpec，用于自定义网格布局
import matplotlib.pyplot as plt                                   # 导入 pyplot，用于绘图
import matplotlib.patches as mpatches                             # 导入 patches，用于画矩形框

fig = plt.figure(figsize=(12, 6))                                 # 创建画布 12x6
gs = GridSpec(1, 3, figure=fig)                                   # 创建 1 行 3 列网格，绑定到 fig

ax1 = fig.add_subplot(gs[0, 0:2])                                 # 左侧子图占第 0 行第 0~1 列（2 列宽）
ax1.imshow(img)                                                   # 左侧显示原图
ax1.set_title(f'Original  {w}x{h}  (occupies 2 columns)')         # 左侧标题
ax1.axis('off')                                                   # 关闭左侧坐标轴

ax2 = fig.add_subplot(gs[0, 2])                                   # 右侧子图占第 0 行第 2 列（1 列宽）
ax2.imshow(small)                                                 # 右侧显示缩小图
ax2.set_title(f'Half  {w//2}x{h//2}  (occupies 1 column)')        # 右侧标题
ax2.axis('off')                                                   # 关闭右侧坐标轴

# ===== 在 figure 坐标系上把 1x3 的三个 cell 边界画出来 =====
for i in range(4):                                                # 画 4 条竖线，即 3 个 cell 的左右边界
    x = i / 3.0                                                   # 每条竖线在 figure 宽度方向的比例位置
    fig.add_artist(plt.Line2D([x, x], [0, 1],                     # 在 figure 坐标 (0~1) 上画竖线
                              transform=fig.transFigure,          # 使用 figure 归一化坐标系
                              color='red', linewidth=1.5,         # 红色、线宽 1.5
                              linestyle='--'))                    # 虚线，便于区分

# ===== 给左右两个子图各画一个矩形框，标出它们实际占的格子 =====
for ax, color, label in [(ax1, 'lime', 'cell 0-1 (2 cols)'),      # 左侧子图：占 2 格，绿色框
                         (ax2, 'orange', 'cell 2 (1 col)')]:      # 右侧子图：占 1 格，橙色框
    bbox = ax.get_position()                                      # 获取该子图在 figure 中的位置矩形
    rect = mpatches.Rectangle((bbox.x0, bbox.y0),                 # 矩形左下角坐标
                              bbox.width, bbox.height,            # 矩形宽高
                              transform=fig.transFigure,          # 使用 figure 归一化坐标系
                              fill=False,                         # 不填充，只画边框
                              edgecolor=color, linewidth=2.5)     # 边框颜色与线宽
    fig.add_artist(rect)                                          # 把矩形加到 figure 上
    ax.text(0.5, -0.08, label, transform=ax.transAxes,            # 在子图下方写标签
            ha='center', va='top', color=color, fontsize=11)      # 居中对齐，颜色与框一致

plt.tight_layout()                                                # 自动调整间距
plt.show()                                                        # 显示图像

print('现象：左边明显比右边大。')                                  # 现象说明
print('原因：GridSpec 让左边占 2 列，物理宽度是右边的 2 倍。')      # 原因说明

##### 实验三：两个独立 figure，不同 figsize

- 同一张 512×512 的图，放进 `figsize=(8,8)` 和 `figsize=(4,4)`；
- 像素数完全没变，但屏幕大小差一倍；
- 说明 **Matplotlib 的显示大小由 figsize 决定**。

In [ ]:
fig1 = plt.figure(figsize=(8, 8))                             # 创建第 1 个 figure，8x8 英寸
ax1 = fig1.add_subplot(111)                                   # 添加 1x1 子图
ax1.imshow(img)                                               # 显示原图
ax1.set_title(f'figsize=(8, 8)  |  image {w}x{h}')            # 设置标题
ax1.axis('off')                                               # 关闭坐标轴

fig2 = plt.figure(figsize=(4, 4))                             # 创建第 2 个 figure，4x4 英寸
ax2 = fig2.add_subplot(111)                                   # 添加 1x1 子图
ax2.imshow(img)                                               # 显示同一张原图
ax2.set_title(f'figsize=(4, 4)  |  image {w}x{h}')            # 设置标题
ax2.axis('off')                                               # 关闭坐标轴

plt.show()                                                    # 显示两个 figure

print('现象：两个窗口大小不同，图像像素数相同。')              # 现象说明
print('原因：显示大小由 figsize 决定，与像素数无关。')        # 原因说明

##### 实验四：OpenCV 窗口验证（如有 GUI 环境）

- 如果有图形界面，可以用 `cv2.imshow` 分别显示 512×512 和 256×256；
- OpenCV 默认按 **1 图像像素 = 1 屏幕像素** 显示；
- 所以两个窗口大小会明显不同，512×512 大约是 256×256 的 2 倍；
- 这直接证明：**OpenCV 窗口大小由像素数决定，而不是由内容决定**。

> 如果没有 GUI 环境，或者是在 Jupyter 里，实验四可能弹不出窗口，此时可以只看前三组结果。

In [ ]:
try:                                                          # 捕获无 GUI 环境下的异常
    cv2.imshow('Original 512x512', img)                       # 显示 512x512 原图
    cv2.imshow('Half 256x256', small)                         # 显示 256x256 缩小图
    print('已请求显示两个 OpenCV 窗口：Original 512x512 和 Half 256x256。')  # 提示信息
    print('现象：512x512 窗口大小约为 256x256 窗口的 2 倍。')  # 现象说明
    print('原因：OpenCV 默认按 1 图像像素 = 1 屏幕像素显示。')  # 原因说明
    print('按任意键关闭窗口...')                              # 提示按键关闭
    cv2.waitKey(0)                                            # 无限等待按键，驱动 GUI 事件循环
    cv2.destroyAllWindows()                                   # 关闭所有 OpenCV 窗口
except Exception as e:                                        # 捕获异常
    print('当前环境无法弹出 OpenCV 窗口，已跳过实验四。')      # 跳过提示
    print('错误信息：', e)                                    # 错误信息
    print('如需验证，请在带图形界面的本地环境中运行本段代码。')  # 环境提示

##### 四个实验汇总

1. **subplots 并排**：左右一样大，因为格子被强制对齐；
2. **GridSpec 宽度不同**：左边占 2 列，明显比右边大；
3. **两个独立 figure + 不同 figsize**：同一张图，大小不同；
4. **OpenCV 窗口**：512×512 约为 256×256 的 2 倍，因为按 1 图像像素 = 1 屏幕像素显示。

**结论**：

- Matplotlib 里，显示大小由 **figure 布局** 决定，不是由图像本身的像素数决定；
- OpenCV 窗口里，显示大小由 **像素数** 决定，默认 1 图像像素 = 1 屏幕像素；
- 无论哪种情况，图像文件里都 **只有像素数**，没有“显示大小”这个字段。

**七、一句话总结**

> 像素在 **数字层面没有物理大小，也没有物理间距**，它只是采样点上的一个数值。
> 只有在 **显示或打印** 时，才根据 PPI / DPI 给像素赋予物理尺寸。
> 显示出来的大小，是 **像素数 ÷ 显示设备的 PPI × 软件缩放比例** 决定的；
> 打印出来的大小，是 **像素数 ÷ 图像 DPI** 决定的。
> “像素是方块”只是显示/插值时的可视化比喻，不是图像的本质。

### 1.3.1 查看文件头部数据

在分层框架里，下面这段属于 **第 1、2 层**：签名 + 元信息。
图像文件开头几个字节是 **magic number / 文件签名**，用来标识格式。

- JPG 开头：`ff d8 ff e0 ... 4a 46 49 46`（`\xff\xd8` 是 SOI，`JFIF` 是标识）
- PNG 开头：`89 50 4e 47 0d 0a 1a 0a`（即 `\x89PNG\r\n\x1a\n`）

下面我们逐字节拆解一段真实的 JPG 头部数据。

In [ ]:
path = '01_Picture/01_cat.jpg'                                # 设置图像路径

with open(path, 'rb') as f:                                   # 以二进制只读方式打开文件
    raw_bytes = f.read()                                      # 一次性读取全部字节

print('文件大小(字节):', len(raw_bytes))                      # 打印文件总字节数
print('前 16 个字节(hex):', raw_bytes[:16].hex(' '))          # 打印前 16 字节的十六进制形式
print('前 16 个字节(repr):', raw_bytes[:16])                  # 打印前 16 字节的字节串形式

### 1.3.2 逐字节解读头部数据

以截图中这段真实的 JPG 头部为例：

```
ff d8 ff e0 00 10 4a 46 49 46 00 01 01 00 00 01
```

对应 repr：

```
b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01'
```

逐段解读：

**1）`ff d8` —— SOI（Start of Image）**
- JPEG 文件的 **magic number**
- 固定值，任何 JPEG 文件都以这两个字节开头
- 作用：告诉解码器“我是一个 JPEG 图像”
- 类似 PNG 的 `89 50 4e 47 0d 0a 1a 0a`

**2）`ff e0` —— APP0 标记（Application Segment 0）**
- JPEG 用 `ff` + 标记字节 表示各种段
- `e0` 表示 **APP0**，通常是 JFIF 应用段
- 后面会跟一个 2 字节的长度字段

**3）`00 10` —— APP0 段长度**
- 大端序（big-endian）表示：`0x0010` = 16
- 意思是 **APP0 段总长度 16 字节**（包含这 2 字节长度本身）
- 所以从 `00 10` 开始往后数 16 字节，就是整个 APP0 段

**4）`4a 46 49 46 00` —— `"JFIF\0"`**
- ASCII 解码：`J F I F \0`
- 表示这是 **JFIF 格式**（JPEG File Interchange Format）
- 最后那个 `00` 是字符串结束符
- 这就是“文件签名”里标识具体格式的部分

**5）`01 01` —— 版本号**
- 第一个 `01`：主版本号 = 1
- 第二个 `01`：次版本号 = 1
- 即 **JFIF 版本 1.1**

**6）`00` —— 密度单位**
- `0` = 无单位（只记录像素宽高比）
- `1` = 每英寸点数（DPI）
- `2` = 每厘米点数（DPCM）

**7）`00 01` —— X 方向密度**
- 大端序：`0x0001` = 1

**8）`00 01` —— Y 方向密度**
- 大端序：`0x0001` = 1

> 前 16 字节刚好切到 `00 01` 的中间，所以看起来像 `00 01 00 00 01`，
> 其实是 X/Y 密度各占 2 字节，后面还有缩略图宽高等字段没显示出来。

### 1.3.3 APP0 段完整结构（共 16 字节）

APP0 段从第 0 字节到第 15 字节，依次是：

- 偏移 0 处是 `ff e0`，表示 APP0 标记；
- 偏移 2 处是 `00 10`，表示段长度等于 16；
- 偏移 4 处是 `4a 46 49 46 00`，即字符串 `JFIF\0`；
- 偏移 9 处是 `01 01`，表示版本 1.1；
- 偏移 11 处是 `00`，表示密度单位是无单位；
- 偏移 12 处是 `00 01`，表示 X 密度等于 1；
- 偏移 14 处是 `00 01`，表示 Y 密度等于 1；
- 偏移 16 处是 `00`，表示缩略图宽等于 0（在截图之外）；
- 偏移 17 处是 `00`，表示缩略图高等于 0（在截图之外）。

### 1.3.4 头部数据到底包含了哪些信息？

在分层框架里，头部数据属于 **第 1、2 层（签名 + 元信息）**，
归纳一下，这段头部告诉了我们：

1. **文件格式**：JPEG（`ff d8`）
2. **具体子格式**：JFIF（`4a 46 49 46`）
3. **JFIF 版本**：1.1
4. **密度单位**：无单位
5. **像素宽高比**：1:1
6. **缩略图**：宽高均为 0（无缩略图）

注意：
- 头部 **不包含** 图像宽高、像素数据、颜色通道信息
- 图像的真实宽高、位深、分量数在 **第 3 层 图像描述段**（SOF 段）里
- 像素数据在 **第 5 层 压缩像素数据**（SOS 段之后）里

### 1.3.5 和其他格式的头部对比

- JPEG 开头是 `ff d8 ff e0 ... JFIF`，表示 SOI 加 APP0/JFIF 标识；
- PNG 开头是 `89 50 4e 47 0d 0a 1a 0a`，是 PNG 签名（8 字节）；
- GIF 开头是 `47 49 46 38`，即 `GIF8`；
- BMP 开头是 `42 4d`，即 `BM`。

一句话总结：

> 头部数据 = **magic number（格式标识）+ 格式版本 + 元信息（密度、缩略图等）**，
> 但 **不包含图像宽高和像素数据**，那些在后面的段里。

### 1.3.6 查看解码后的像素数据

用 `IMREAD_UNCHANGED` 读进来，看到的就是最接近文件本身的解码结果。
注意彩色图的每个像素是 **BGR** 顺序，不是 RGB。

In [ ]:
img = cv2.imread(path, cv2.IMREAD_UNCHANGED)                  # 以原样模式读取，得到最接近文件的解码结果

print('shape:', img.shape)                                    # 打印数组形状 (高, 宽, 通道数)
print('dtype:', img.dtype)                                    # 打印数据类型，通常为 uint8
print('最小/最大值:', img.min(), img.max())                  # 打印像素最小值与最大值
print('前 3 行前 3 列像素(BGR):')                             # 打印提示信息
print(img[:3, :3])                                            # 打印左上角 3x3 区域像素值

### 1.3.7 对比不同 flags 下的数据

同一份文件，三种 `flags` 读出来是三个不同的数组。
如果原图是带 alpha 的 PNG，`IMREAD_UNCHANGED` 会返回 4 通道。

In [ ]:
path = '01_Picture/01_cat.jpg'                                # 设置图像路径
img_color     = cv2.imread(path, cv2.IMREAD_COLOR)             # 以彩色模式读取，返回 3 通道 BGR
img_gray      = cv2.imread(path, cv2.IMREAD_GRAYSCALE)         # 以灰度模式读取，返回单通道灰度
img_unchanged = cv2.imread(path, cv2.IMREAD_UNCHANGED)         # 以原样模式读取，保留全部通道

print('COLOR     shape:', img_color.shape,     'dtype:', img_color.dtype)      # 打印彩色图形状与类型
print('GRAYSCALE shape:', img_gray.shape,      'dtype:', img_gray.dtype)       # 打印灰度图形状与类型
print('UNCHANGED shape:', img_unchanged.shape, 'dtype:', img_unchanged.dtype)  # 打印原样图形状与类型

y, x = 100, 100                                               # 设置采样点坐标
print('COLOR    BGR:', img_color[y, x])                       # 打印彩色图采样点 BGR
print('GRAY     :',    img_gray[y, x])                        # 打印灰度图采样点灰度值
print('UNCHANGED BGR(A):', img_unchanged[y, x])               # 打印原样图采样点 BGR(A)

### 1.3.8 原始字节 vs 解码数据

在分层框架里：
- **原始字节**：第 5 层“压缩像素数据”，人眼无法直接对应到像素
- **解码后**：`(H, W, C)` 的 numpy 数组

`cv2.imread()` 做的就是：**原始字节 → 按 flags 解码 → numpy 数组**。

In [ ]:
print('原始字节前 32 个:', raw_bytes[:32])                    # 打印原始文件前 32 字节（压缩编码）

print('解码后像素(前 2x2):')                                  # 打印提示信息
print(img_color[:2, :2])                                      # 打印彩色图左上角 2x2 像素值

### 1.3.9 层次小结

按层次看：

- 第一层文件签名，内容是 magic number，例如 `ff d8`、`89 50 4e 47`；
- 第二层元信息，内容是版本、密度、缩略图、EXIF，例如 `JFIF 1.1、120 DPI`；
- 第三层图像描述，内容是宽高、位深、分量数，例如 `512×512, 8bit, 1 分量`；
- 第四层辅助表，内容是量化表、霍夫曼表、调色板，例如 DQT / DHT / PLTE；
- 第五层压缩像素数据，内容是编码后的数据，例如 DCT 系数 / Deflate；
- 第六层结束标记，内容是文件结束，例如 `ff d9` / IEND。

而 OpenCV 读出来的 numpy 数组，是第 3、5 层综合后的结果：

- 文件头部是 magic number、格式标识、元信息，例如 `ff d8 ff e0 ... JFIF`；
- 原始字节是压缩后的二进制流，例如 `b'\xff\xd8\xff\xe0...'`；
- 解码数据是 numpy 数组，例如 `shape=(414,500,3), dtype=uint8`；
- flags 影响的是通道数、颜色空间、alpha，对应 COLOR / GRAYSCALE / UNCHANGED。

这样就把 **“原始数据固定 → flags 决定解码输出形态”** 这件事，从字节层到数组层完整看清楚了。

### 1.3.10 灰度图像与彩色图像的头部差异

前面的头部讲解以彩色 JPG 为例，这里补充灰度的情况。
首先要区分两个容易混淆的概念：

- **真·灰度 JPEG**：文件里就是单通道，通道数为 1，常见于相机黑白模式、专业灰度扫描；
- **彩色 JPEG 用 `IMREAD_GRAYSCALE` 读**：文件里仍是彩色，只是读出来变成 1 通道。

也就是说：
- `cv2.imread(path, cv2.IMREAD_GRAYSCALE)` 属于第二种——文件还是彩色 JPG，只是读出来转成了单通道。
- “灰度图像的信息”如果指的是**真·灰度 JPEG 文件**，那头部确实和彩色不同。

#### 相同部分

- `ff d8`：SOI，所有 JPEG 都一样
- `ff e0 ... JFIF`：APP0 段，格式标识，基本一样
- `ff db`：DQT，量化表
- `ff c0` / `ff c2`：SOF，帧头
- `ff da`：SOS，扫描数据开始
- `ff d9`：EOI，文件结束

#### 关键差异：SOF 段（`ff c0` / `ff c2`）

SOF 段里有一个字段叫 **Number of Components（分量数）**：

- 彩色 JPEG 的分量数是 `03`，表示 Y、Cb、Cr 三个分量；
- 灰度 JPEG 的分量数是 `01`，表示只有 Y 一个分量。

这就是文件层面最本质的区别。

#### 其他可能差异

- APP0/JFIF：彩色有，灰度有或可能没有；
- DQT 量化表：彩色通常 2 张（Y 和 C），灰度通常 1 张；
- SOF 分量数：彩色 3，灰度 1；
- DHT 霍夫曼表：彩色更多，灰度更少；
- SOS 扫描：彩色可能多次扫描，灰度通常一次扫描。

#### 彩色图读成灰度呢？

```python
img = cv2.imread('cat.jpg', cv2.IMREAD_GRAYSCALE)
```

- **文件仍然是彩色 JPEG**，头部和彩色完全一样
- OpenCV 在 **解码后**，把 YCbCr 里的 Y 通道取出来（或按亮度公式算），返回单通道数组
- 这种情况看头部 → 和彩色一样；看解码结果 → shape 是 `(H, W)`

#### 自己验证：找 SOF 段

```python
import re
for m in re.finditer(b'\xff[\xc0\xc2]', raw_bytes):
    i = m.start()
    print('SOF at', i, ':', raw_bytes[i:i+12].hex(' '))
    print('  分量数 =', raw_bytes[i+9])
```

彩色 JPG 会打印 `分量数 = 3`，真灰度 JPG 会打印 `分量数 = 1`。

> 注意：文件前 32 字节里通常看不到 SOF 段，
> 因为 SOI + APP0/JFIF（18 字节）+ DQT（通常 67 字节以上）就已经超过 32 字节了。
> 所以位深、宽高、分量数都不在前 32 字节里，得往后找 `ff c0` / `ff c2`。

### 1.3.11 灰色深度（位深）在头部哪里

灰度图像不仅要告知“我是灰度”，还要告知“每个像素用多少位表示”。
这个信息专业叫 **灰色深度（bit depth）** 或 **精度（Sample Precision）**，
就在 **SOF 段的“精度”字段**里。

常见位深：

- 8 bit 对应 256 个灰度级，取值范围 0~255，最常见；
- 10 bit 对应 1024 个灰度级，取值范围 0~1023，医学影像、专业相机常用；
- 12 bit 对应 4096 个灰度级，取值范围 0~4095，高端医疗、科研常用；
- 16 bit 对应 65536 个灰度级，取值范围 0~65535，遥感、医学 DICOM 常用；
- 1 bit 对应 2 个灰度级，取值 0/1，是二值图。

彩色图同样有位深，通常说的是 **每个通道** 的位深，比如 8bit/通道 = 24bit 真彩色。

需要强调的是：**位深决定的是灰度级数（也就是取值范围），并不决定像素的物理大小或显示大小**。
像素在数字层面没有物理尺寸，物理大小只在显示/打印环节由 PPI / DPI 决定，和位深无关。

#### SOF 段的精确结构

```
ff c0  00 11  08  01 9e  02 58  03  ...
↑      ↑      ↑   ↑      ↑      ↑
标记   长度   精度 高     宽     分量数
```

其中：
- 段起始偏移记为 `sof_offset`
- `raw[sof_offset : sof_offset+2]` → `ff c0` 标记
- `raw[sof_offset+2 : sof_offset+4]` → 段长度
- **`raw[sof_offset + 4]` → 精度 = 位深** ← 就在这里
- `raw[sof_offset+5 : sof_offset+7]` → 高度
- `raw[sof_offset+7 : sof_offset+9]` → 宽度
- `raw[sof_offset + 9]` → 分量数

**精度字段紧跟在段长度后面，彩色灰度都有它，不是灰度专有。**

#### 两种定位位深的方法

**方法 1：用段扫描器**

```python
for offset, marker, length, payload in scan_jpeg_segments(raw):
    if marker in (0xC0, 0xC1, 0xC2, 0xC3):
        precision = payload[0]   # ← 位深
        print('位深 =', precision, 'bit')
```

**方法 2：直接按偏移取**

```python
idx = raw.find(b'\xff\xc0')          # 定位 SOF0
bit_depth = raw[idx + 4]             # ← 位深
print('SOF0 偏移 =', idx, ' 位深 =', bit_depth, 'bit')
```

#### 为什么必须告知灰色深度

1. **解码器要知道每个样本占多少位**，才能正确切分比特流
2. **动态范围不同**，后续显示/计算方式不同
   - 8bit → 直接映射到 0~255
   - 12bit → 需要右移或归一化才能显示
3. **压缩表（DHT/DQT）也和位深配合**，位深不同，霍夫曼表设计不同

#### 和 OpenCV 的关系

`cv2.imread()` 默认只输出 **uint8**：

- 8bit 灰度 → `uint8`，shape `(H, W)`
- 12/16bit 灰度 → OpenCV 可能 **截断或缩放** 成 8bit
- 要保留原始位深，得用 `IMREAD_UNCHANGED` 或 `IMREAD_ANYDEPTH`

```python
img = cv2.imread('gray16.png', cv2.IMREAD_UNCHANGED)
print(img.dtype)   # uint16
print(img.max())   # 65535

img = cv2.imread('gray16.png', cv2.IMREAD_GRAYSCALE)
print(img.dtype)   # uint8，位深被压缩了
```

#### 归纳：灰度图像头部要告诉解码器什么

- 是灰度还是彩色，在 **第 3 层 图像描述** 的 SOF 分量数里，1 表示灰度，3 表示彩色；
- 灰色深度，也在 **第 3 层 图像描述** 的 SOF 精度里，取值为 8/12/16 bit；
- 图像宽高，在 **第 3 层 图像描述** 的 SOF 高/宽里，决定数组形状；
- 压缩方式，在 **第 3 层 图像描述** 的 SOF 标记类型里，可能为基线/渐进/无损；
- 量化表，在 **第 4 层 辅助表** 的 DQT 里，用于反量化；
- 霍夫曼表，在 **第 4 层 辅助表** 的 DHT 里，用于熵解码。

其中 **分量数 + 精度** 一起，才完整描述了“这是几位灰度图”。

#### 别混淆：APP0 里的密度不是位深

APP0/JFIF 段属于 **第 2 层 元信息**，里面的 `00 78 00 78` 是 **X/Y 方向密度（DPI）**，
不是位深，别搞混。位深只在 **第 3 层 图像描述段**（SOF）里出现一次。

### 1.3.12 实战：分析灰度图 `bone.jpg` 的头部与位深

下面是一段 **完整可运行** 的代码，用来分析 `bone.jpg` 的头部信息，
并 **准确定位到位深**。

In [ ]:
import struct                                                 # 导入 struct，用于按大端序解析二进制字段

path = '01_Picture/bone.jpg'                                  # 设置图像路径

with open(path, 'rb') as f:                                   # 以二进制只读方式打开文件
    raw = f.read()                                            # 读取全部字节到 raw

print('=' * 60)                                               # 打印分隔线
print('文件大小(字节):', len(raw))                            # 打印文件大小
print('前 32 字节(hex):', raw[:32].hex(' '))                  # 打印前 32 字节的十六进制形式
print('前 32 字节(repr):', raw[:32])                          # 打印前 32 字节的字节串形式
print('=' * 60)                                               # 打印分隔线


def scan_jpeg_segments(data):                                 # 定义 JPEG 段扫描函数
    """
    遍历 JPEG 的所有段。
    返回 (offset, marker, length, payload)：
      - offset : 段起始偏移
      - marker : 段标记字节（不含 0xFF）
      - length : 段长度（含长度字段自身的 2 字节）
      - payload: 段内容（不含标记和长度字段）
    """
    i = 0                                                     # 初始化索引 i
    n = len(data)                                             # 获取数据总长度
    while i < n - 1:                                          # 当索引未到达末尾时循环
        if data[i] != 0xFF:                                   # 当前字节不是 0xFF，则不是段标记
            i += 1                                            # 索引后移
            continue                                          # 继续循环
        marker = data[i + 1]                                  # 读取段标记字节
        if marker in (0x00, 0xFF, 0xD8, 0xD9) or 0xD0 <= marker <= 0xD7:  # 填充字节 / 独立标记 / 重启标记
            i += 2                                            # 跳过这两个字节
            continue                                          # 继续循环
        if i + 4 > n:                                         # 剩余字节不足以读取长度字段
            break                                             # 结束循环
        length = struct.unpack('>H', data[i + 2:i + 4])[0]    # 大端序解析 2 字节段长度
        payload = data[i + 4:i + 2 + length]                  # 截取段内容
        yield (i, marker, length, payload)                    # 产出当前段信息
        if marker == 0xDA:                                    # 遇到 SOS 段
            break                                             # 停止扫描
        i += 2 + length                                       # 移动到下一段


marker_names = {                                              # 定义段标记名称映射
    0xE0: 'APP0/JFIF  (元信息)',                              # APP0 段
    0xE1: 'APP1/EXIF  (元信息)',                              # APP1 段
    0xDB: 'DQT  量化表 (辅助表)',                             # DQT 段
    0xC0: 'SOF0 基线  (图像描述)',                            # SOF0 段
    0xC1: 'SOF1 扩展  (图像描述)',                            # SOF1 段
    0xC2: 'SOF2 渐进  (图像描述)',                            # SOF2 段
    0xC3: 'SOF3 无损  (图像描述)',                            # SOF3 段
    0xC4: 'DHT  霍夫曼表 (辅助表)',                           # DHT 段
    0xDD: 'DRI  重启间隔',                                    # DRI 段
    0xDA: 'SOS  扫描开始 (像素数据)',                         # SOS 段
    0xD9: 'EOI  文件结束',                                    # EOI 段
}

print('\n所有 JPEG 段（按分层标注）：')                      # 打印提示信息
print('-' * 60)                                               # 打印分隔线
for offset, marker, length, payload in scan_jpeg_segments(raw):  # 遍历所有段
    name = marker_names.get(marker, f'其他(FF{marker:02X})')  # 获取段名称
    print(f'offset={offset:6d}  marker=FF{marker:02X}  len={length:5d}  {name}')  # 打印段信息
print('-' * 60)                                               # 打印分隔线


def parse_sof(payload):                                       # 定义解析 SOF 段的函数
    """
    SOF payload 结构：
      payload[0]      精度 (bit depth)
      payload[1:3]    高
      payload[3:5]    宽
      payload[5]      分量数
      payload[6:]     每个分量 3 字节（ID, 采样因子, 量化表）
    """
    precision = payload[0]                                    # 读取精度（位深）
    height = struct.unpack('>H', payload[1:3])[0]             # 大端序读取高度
    width  = struct.unpack('>H', payload[3:5])[0]             # 大端序读取宽度
    ncomp  = payload[5]                                       # 读取分量数
    comps = []                                                # 初始化分量列表
    for k in range(ncomp):                                    # 遍历每个分量
        cid    = payload[6 + 3 * k]                           # 分量 ID
        samp   = payload[7 + 3 * k]                           # 采样因子
        qtable = payload[8 + 3 * k]                           # 量化表 ID
        comps.append((cid, samp, qtable))                     # 添加到分量列表
    return precision, height, width, ncomp, comps             # 返回解析结果


print('\nSOF 段解析（含位深）：')                            # 打印提示信息
print('-' * 60)                                               # 打印分隔线

found_sof = False                                             # 标记是否找到 SOF 段
for offset, marker, length, payload in scan_jpeg_segments(raw):  # 遍历所有段
    if marker in (0xC0, 0xC1, 0xC2, 0xC3):                    # 判断是否为 SOF 段
        found_sof = True                                      # 标记找到 SOF 段
        precision, height, width, ncomp, comps = parse_sof(payload)  # 解析 SOF 段
        print(f'SOF 段偏移 = {offset}')                       # 打印 SOF 段偏移
        print(f'SOF 标记 = FF{marker:02X}')                   # 打印 SOF 标记
        print(f'段长度 = {length}')                           # 打印段长度
        print(f'位深(精度) = {precision} bit')                # 打印位深
        print(f'高度 = {height}')                             # 打印高度
        print(f'宽度 = {width}')                              # 打印宽度
        print(f'分量数 = {ncomp}  ({ "灰度" if ncomp == 1 else "彩色" })')  # 打印分量数并判断灰度/彩色
        print('每个分量：')                                   # 打印提示信息
        for cid, samp, q in comps:                            # 遍历每个分量
            print(f'  分量ID={cid}  采样因子=0x{samp:02X}  量化表={q}')  # 打印分量信息
        sof_offset = offset                                   # 记录 SOF 段偏移
        bit_depth = raw[sof_offset + 4]                       # 直接按偏移读取位深
        print(f'\n[直接验证] raw[{sof_offset}+4] = raw[{sof_offset + 4}] = {bit_depth} bit')  # 打印验证结果

if not found_sof:                                             # 如果未找到 SOF 段
    print('未找到 SOF 段')                                    # 打印提示信息


try:                                                          # 尝试导入 OpenCV
    import cv2                                                # 导入 OpenCV
    print('\nOpenCV 交叉验证：')                              # 打印提示信息
    print('-' * 60)                                           # 打印分隔线
    img_unchanged = cv2.imread(path, cv2.IMREAD_UNCHANGED)    # 以原样模式读取图像
    if img_unchanged is not None:                             # 读取成功
        print('IMREAD_UNCHANGED shape:', img_unchanged.shape) # 打印形状
        print('IMREAD_UNCHANGED dtype:', img_unchanged.dtype) # 打印数据类型
    else:                                                     # 读取失败
        print('IMREAD_UNCHANGED 读取失败')                    # 打印失败信息
    img_color = cv2.imread(path, cv2.IMREAD_COLOR)            # 以彩色模式读取图像
    if img_color is not None:                                 # 读取成功
        print('IMREAD_COLOR     shape:', img_color.shape)     # 打印形状
except ImportError:                                           # 未安装 OpenCV
    print('\n未安装 OpenCV，跳过交叉验证')                    # 打印提示信息

运行后的典型输出（对真灰度 `bone.jpg`）：

```
============================================================
文件大小(字节): 18067
前 32 字节(hex): ff d8 ff e0 00 10 4a 46 49 46 00 01 01 01 00 78 00 78 00 00 ff db 00 43 00 07 05 05 06 05 04 07
前 32 字节(repr): b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00x\x00x\x00\x00\xff\xdb\x00C\x00\x07\x05\x05\x06\x05\x04\x07'
============================================================

所有 JPEG 段（按分层标注）：
------------------------------------------------------------
offset=     0  marker=FFD8  len=    0  其他(FFD8)
offset=     2  marker=FFE0  len=   16  APP0/JFIF  (元信息)
offset=    20  marker=FFDB  len=   67  DQT  量化表 (辅助表)
offset=    89  marker=FFC0  len=   11  SOF0 基线  (图像描述)
offset=   102  marker=FFC4  len=   ...  DHT  霍夫曼表 (辅助表)
offset=   ...  marker=FFDA  len=   ...  SOS  扫描开始 (像素数据)
------------------------------------------------------------

SOF 段解析（含位深）：
------------------------------------------------------------
SOF 段偏移 = 89
SOF 标记 = FFC0
段长度 = 11
位深(精度) = 8 bit
高度 = 512
宽度 = 512
分量数 = 1  (灰度)
每个分量：
  分量ID=1  采样因子=0x11  量化表=0

[直接验证] raw[89+4] = raw[93] = 8 bit

OpenCV 交叉验证：
------------------------------------------------------------
IMREAD_UNCHANGED shape: (512, 512)
IMREAD_UNCHANGED dtype: uint8
IMREAD_COLOR     shape: (512, 512, 3)
```

#### 从分层视角看 `bone.jpg`

- 第一层文件签名，对应 SOI `ff d8`，表示这是 JPEG；
- 第二层元信息，对应 APP0/JFIF，记录了版本 1.1 和密度 120 DPI；
- 第三层图像描述，对应 SOF0 `ff c0`，记录了位深 8bit、512×512、分量数 1；
- 第四层辅助表，对应 DQT / DHT，灰度只有 1 张量化表；
- 第五层像素数据，对应 SOS `ff da` 之后，是压缩的 DCT 系数；
- 第六层结束标记，对应 EOI `ff d9`，表示文件结束。

#### `bone.jpg` 头部信息解读

- SOI 为 `ff d8`，说明是 JPEG 文件；
- APP0 为 `ff e0 ... JFIF`，说明是 JFIF 格式；
- JFIF 版本为 `01 01`，即 1.1；
- 密度单位为 `01`，表示每英寸点数（DPI）；
- X/Y 密度为 `00 78 00 78`，即 120 DPI × 120 DPI；
- DQT 为 `ff db 00 43 ...`，只有 1 张量化表，因为灰度只需 1 张；
- SOF0 标记为 `ff c0`，表示基线 DCT 编码；
- 位深(精度)为 `8`，即 8 bit，取值 0~255；
- 高/宽为 `512 × 512`，写在 SOF 段里，单位是像素；
- 分量数为 `1`，说明是真·灰度图，只有 Y 分量；
- 分量采样为 `0x11`，表示无下采样。

> 注意：512×512 指的是像素数，不是物理尺寸。
> 这个文件同时记录了 120 DPI，所以打印物理尺寸约为 `512 / 120 ≈ 4.27 英寸`。
> 而显示大小则取决于显示设备的 PPI 和软件缩放，和 DPI 无关。

#### 关键点回顾

- 文件格式在第 1 层签名里，位置是开头 2 字节，代码上用 `raw[:2]`；
- JFIF 版本在第 2 层元信息里，位置是 APP0 段，代码上用 `raw[11:13]`；
- 密度单位在第 2 层元信息里，位置是 APP0 段，代码上用 `raw[13]`；
- X/Y 密度在第 2 层元信息里，位置是 APP0 段，代码上用 `raw[14:18]`；
- 位深在第 3 层图像描述里，位置是 SOF 段、段长之后第 1 字节，代码上用 `raw[sof_offset + 4]`；
- 高度在第 3 层图像描述里，位置是 SOF 段，代码上用 `struct.unpack('>H', raw[sof_offset+5:sof_offset+7])`；
- 宽度在第 3 层图像描述里，位置是 SOF 段，代码上用 `struct.unpack('>H', raw[sof_offset+7:sof_offset+9])`；
- 分量数在第 3 层图像描述里，位置是 SOF 段，代码上用 `raw[sof_offset + 9]`；
- 是灰度还是彩色由分量数决定，1 表示灰度，3 表示彩色。

> **一句话**：位深就在 **SOF 段里，段长度字段之后第 1 个字节**，
> 代码上用 `raw[sof_offset + 4]` 直接取。

## 1.4 读取失败与路径检查

如果 `cv2.imread()` 返回 `None`，后续 `cv2.imshow()` 会报错。
因此工程中通常需要判断：

```python
if img is None:
    raise FileNotFoundError('图像读取失败，请检查路径')
```

另外，OpenCV 对中文路径支持不稳定。若路径中包含中文，建议使用：

```python
img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
```

In [ ]:
bad_img = cv2.imread('not_exist.jpg')                         # 尝试读取不存在的文件，返回 None
print(bad_img)                                                # 输出 None，表示读取失败

## 1.5 灰度图读取

使用 `cv2.IMREAD_GRAYSCALE` 可以直接读取为单通道灰度图。
此时 `img.shape` 为 `(高度, 宽度)`，没有第三维通道。

注意：
- 如果文件本身就是灰度（比如 `bone.jpg`），这里的“灰度”就是文件里的 Y 分量。
- 如果文件是彩色（比如 `01_cat.jpg`），这里的“灰度”是解码时按亮度加权公式现场计算出来的。

In [ ]:
img_gray = cv2.imread('01_Picture/01_cat.jpg', cv2.IMREAD_GRAYSCALE)  # 以灰度模式读取彩色图像
print(img_gray.shape if img_gray is not None else '读取失败')         # 成功打印 (H, W)，失败打印提示

# 2. 图像显示

## 2.1 OpenCV 显示机制

OpenCV 的图像显示依赖 GUI 窗口系统，核心函数有三个：

1. `cv2.imshow(winname, img)`：创建或更新一个窗口并显示图像。
2. `cv2.waitKey(delay)`：等待键盘事件，同时驱动 GUI 事件循环。
3. `cv2.destroyAllWindows()`：销毁所有 OpenCV 创建的窗口。

重要理解：
- `cv2.imshow()` 本身不会阻塞程序，它只是把图像提交给窗口系统。
- `cv2.waitKey()` 才是让窗口保持显示、响应按键的关键。
- 如果 `waitKey(0)`，表示无限等待，直到用户按下任意键。
- 如果 `waitKey(5000)`，表示最多等待 5000 毫秒，超时后继续执行。
- 在 Jupyter Notebook 中，OpenCV 窗口有时无法正常弹出，通常与后端或远程环境有关。

## 2.2 图像显示(普通方法)

注意：OpenCV 默认读取格式是 **BGR**，而 Matplotlib 等库通常使用 **RGB**。
- 用 OpenCV 读取 + OpenCV 显示：不需要通道转换。
- 用 OpenCV 读取 + Matplotlib 显示：需要将 BGR 转为 RGB，否则颜色会异常。

In [ ]:
# opencv 默认读取格式是 BGR 格式，matplotlib 或其他库的读取格式可能是 RGB 的
# opencv 读取并用 opencv 自带的展示函数不需要进行通道转换，但 opencv 读取后用其他库展示图片需要通道转换

# 图像显示时,可以创建多个窗口

# 第一个入口参数为展示图像窗口的名字
# 第二个入口参数为展示图像窗口中所展示的图像
img = cv2.imread('01_Picture/01_cat.jpg')                     # 读取图像，默认彩色 BGR
cv2.imshow('image_cat', img)                                  # 创建名为 image_cat 的窗口并显示图像

# 等待时间，毫秒级，0表示任意键终止，5000ms表示5s
cv2.waitKey(5000)                                             # 等待 5000 毫秒，期间窗口保持显示

# 销毁图像窗口
cv2.destroyAllWindows()                                       # 关闭所有 OpenCV 创建的窗口

## 2.3 BGR 与 RGB 转换

OpenCV 读取的彩色图像通道顺序为 BGR。
如果要用 Matplotlib 正确显示，需要转换：

```python
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)
```

也可以手动反转通道：

```python
img_rgb = img[:, :, ::-1]
```

In [ ]:
# 用 Matplotlib 显示时，需要 BGR -> RGB
img = cv2.imread('01_Picture/01_cat.jpg')                     # 读取图像
if img is not None:                                           # 如果读取成功
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # 将 BGR 转为 RGB
    plt.imshow(img_rgb)                                       # 使用 Matplotlib 显示 RGB 图像
    plt.axis('off')                                           # 关闭坐标轴
    plt.show()                                                # 显示图像

## 2.4 图像显示(函数方法)

把显示逻辑封装成函数，可以避免重复写 `imshow`、`waitKey`、`destroyAllWindows`。

封装函数时要注意：
- `cv2.waitKey(0)` 会阻塞，适合单张图调试。
- 如果要连续显示多张图，建议使用较小的等待时间，或改用 Matplotlib 子图。
- 在 Jupyter 中，如果窗口没有正常关闭，可能需要手动重启内核。

In [ ]:
# 绘图显示(封装函数)
def cv_show(name, img):                                       # 定义显示函数，接收窗口名和图像
    """
    使用 OpenCV 显示图像。
    :param name: 窗口名称
    :param img: 图像数组
    """
    cv2.imshow(name, img)                                     # 显示图像
    cv2.waitKey(0)                                            # 无限等待，直到用户按下任意键
    cv2.destroyAllWindows()                                   # 关闭所有窗口

img = cv2.imread('01_Picture/01_cat.jpg')                     # 读取图像
if img is not None:                                           # 如果读取成功
    cv_show('image_cat', img)                                 # 调用封装函数显示图像

## 2.5 多窗口显示示例

OpenCV 允许同时创建多个窗口。
窗口名称必须不同，否则后一个会覆盖前一个。

In [ ]:
img = cv2.imread('01_Picture/01_cat.jpg')                     # 读取彩色图像
img_gray = cv2.imread('01_Picture/01_cat.jpg', cv2.IMREAD_GRAYSCALE)  # 读取灰度图像

if img is not None and img_gray is not None:                  # 如果两幅图都读取成功
    cv2.imshow('color', img)                                  # 显示彩色图窗口
    cv2.imshow('gray', img_gray)                              # 显示灰度图窗口
    cv2.waitKey(0)                                            # 无限等待按键
    cv2.destroyAllWindows()                                   # 关闭所有窗口

# 3. 常见问题与注意事项

1. **图像读取失败**：路径错误、文件不存在、中文路径、权限不足。
2. **颜色异常**：OpenCV 是 BGR，Matplotlib 是 RGB，需要转换。
3. **窗口一闪而过**：忘记调用 `cv2.waitKey()`。
4. **Jupyter 中窗口无法显示**：远程服务器、无 GUI 环境、后端冲突。
5. **`cv2.imshow()` 报错**：图像为 `None`，或数据类型不是 `uint8`。
6. **灰度图 shape**：灰度图没有第三维，彩色图有 3 个通道。
7. **窗口名称重复**：同名窗口会互相覆盖。
8. **误解 `flags`**：`flags` 不改变原文件，它决定的是解码后返回数组的通道数、颜色空间与是否保留 alpha；灰度是读取时按亮度公式现场计算的，不是文件里预存的。
9. **混淆原始字节与解码数据**：原始文件字节是压缩编码，无法直接对应像素；解码后的 numpy 数组才是 `(H, W, C)` 的像素矩阵。
10. **误解头部数据**：文件头部只包含 magic number、格式版本、密度、缩略图等元信息，**不包含图像宽高和像素数据**；真实宽高在 SOF 段，像素数据在 SOS 段之后。
11. **混淆“文件灰度”与“读取灰度”**：真灰度 JPEG 的 SOF 分量数 = 1；彩色 JPEG 用 `IMREAD_GRAYSCALE` 读，文件头部和彩色完全一样，只是解码后返回单通道。两者不能混为一谈。
12. **以为精度字段是灰度专有**：SOF 里的精度字段彩色灰度都有，它描述的是每个样本的位深（8/12/16 bit），不是灰度图独有。
13. **忽略位深差异**：`cv2.imread()` 默认输出 uint8，12/16bit 灰度会被压缩；要保留原始位深需用 `IMREAD_UNCHANGED` 或 `IMREAD_ANYDEPTH`。
14. **把 APP0 里的密度当成位深**：APP0/JFIF 段的 `00 78 00 78` 是 X/Y 方向密度（DPI），不是位深；位深只在 SOF 段里出现一次。
15. **在前 32 字节里找位深**：SOI + APP0 + DQT 通常已经超过 32 字节，位深在更靠后的 SOF 段，直接看前 32 字节是看不到的。
16. **把三层信息混为一谈**：元信息（版本、DPI、EXIF）≠ 图像描述（宽高、位深、分量数）≠ 像素数据（压缩编码）。找信息时先判断在哪一层，再定位到具体段。
17. **把像素当成有面积的小方块**：像素在数字层面只是采样点的值，没有物理大小，也没有物理间距；“像素是方块”只是显示/插值时的可视化比喻。
18. **把 DPI 当成显示大小**：DPI 只是文件里的元信息，用于打印和排版；屏幕显示大小由像素数、屏幕 PPI 和软件缩放比例共同决定，DPI 通常不生效。
19. **把位深当成物理尺寸**：位深只决定灰度级数（取值范围），不决定像素物理大小，也不决定显示大小。
20. **误解 alpha 的作用**：alpha 不是“图像是否有透明度”这种二元开关，它是一个有精度的通道，用来按比例混合前景和背景。
21. **以为看到军绿色背景就是“背景色”**：军绿色是文件里透明区域（alpha=0）保存的 BGR 值，不是 figure 底色，也不是图像的真实背景。要真正让透明区域消失，必须手动做一次“前景 × alpha + 背景 × (1-alpha)”的合成。

# 4. 小结

- `cv2.imread()` 返回 `numpy.ndarray` 或 `None`。
- 原始文件数据固定，`flags` 决定解码后输出成什么通道数、什么颜色空间、是否保留 alpha。
- 原始图像文件分层：**签名 + 元信息 + 图像描述 + 辅助表 + 压缩像素数据 + 结束标记**。
- 元信息（版本、DPI、EXIF）≠ 图像描述（宽高、位深、分量数）≠ 像素数据（压缩编码）。
- 宽高的单位是 **像素**，不是物理尺寸；想要物理尺寸，需要结合 DPI / PPI 换算。
- 像素在数字层面 **没有物理大小，也没有物理间距**，只有显示/打印时才根据 PPI / DPI 赋予物理尺寸。
- 屏幕显示大小 ≈ 像素数 ÷ 屏幕 PPI × 软件缩放比例；打印大小 ≈ 像素数 ÷ 图像 DPI。
- 真灰度 JPEG 与彩色 JPEG 的头部差异集中在 **SOF 分量数**：1 vs 3；灰度文件 DQT/DHT 更少、体积更小。
- 彩色 JPEG 用 `IMREAD_GRAYSCALE` 读，头部仍是彩色，只是解码输出单通道。
- 灰色深度（位深）写在 **SOF 精度字段** 里，位置是段长之后第 1 字节，代码上用 `raw[sof_offset + 4]` 直接取；彩色灰度都有它，常见 8bit，专业领域有 10/12/16bit。
- 位深决定灰度级数（取值范围），不决定像素物理大小或显示大小。
- APP0 里的密度（DPI）≠ 位深，不要混淆。
- alpha 是有精度的通道，不是“有/没有”的开关，用于按比例混合前景和背景。
- OpenCV 默认读取为 BGR，Matplotlib 默认按 RGB 显示。
- `cv2.imshow()` 负责显示，`cv2.waitKey()` 负责事件循环，`cv2.destroyAllWindows()` 负责清理窗口。
- 工程中建议封装 `cv_show()`，并检查图像是否读取成功。
- 在 Jupyter 中，OpenCV 窗口显示可能不稳定，必要时可改用 Matplotlib 显示。
- **看到军绿色背景 ≠ 图像背景是军绿色**：它是文件里透明区域保存的 BGR 值，只有手动做 alpha 合成，才能让透明区域真正“消失”。

# 附录

## 附录 A：透明度与 alpha 通道

### A.1 alpha 通道是什么

普通彩色图有 3 个通道：R（红）、G（绿）、B（蓝），决定颜色。
带透明度的图会多出第 4 个通道：A（alpha），决定 **这个像素的透明程度**。

alpha 的取值（以 8bit 为例，0~255）：

- `0` 表示完全透明，什么都看不见；
- `255` 表示完全不透明，正常显示；
- 中间值表示半透明，颜色和背景按比例混合。

所以一张带透明度的图，常见的是 4 通道：RGBA（或 OpenCV 里的 BGRA）。

### A.2 alpha 有什么用

它主要用在 **图层叠加** 场景：

- PNG 图标、logo，背景是透明的；
- 网页、PPT 里把图标贴到任意背景上，透明区域会透出背景；
- 视频字幕、贴纸、游戏精灵图；
- 图像合成，比如把前景人物抠出来贴到另一张图上。

如果 **没有 alpha**，这些“透明区域”就只能用一种固定颜色（比如白色或黑色）表示，贴到别的背景上会显得很生硬。

### A.3 alpha 的位深

alpha 通道也有位深：

- 8bit alpha：0~255，最常见；
- 16bit alpha：0~65535，专业图像；
- 1bit alpha：只有 0 或 1，相当于“要么透明要么不透明”。

所以 alpha 不是“有/没有”的二元概念，它本身也是一个有精度的通道。

### A.4 为什么 `IMREAD_COLOR` 会“忽略透明度”

`IMREAD_COLOR` 在解码时只保留 3 个通道，把 alpha 直接丢弃。
所以读进来的数组是 `(H, W, 3)`，没有第 4 个通道，后续想用 alpha 也没有了。

**1）IMREAD_COLOR（默认）**

- 读进来只有 3 通道 BGR；
- **如果文件里有 alpha，会被直接丢弃**；
- 透明区域按某种约定处理（比如变黑、变白，或和背景混合，取决于格式和解码器）。

**2）IMREAD_UNCHANGED**

- 文件里有什么就还原什么；
- 带 alpha 的 PNG → 返回 4 通道 BGRA；
- 这样你才能自己拿到 alpha 去做合成。

下面用一段可运行代码验证：读一张带 alpha 的 PNG，看 `IMREAD_UNCHANGED` 是否返回 4 通道。

In [ ]:
# A.4 验证：带 alpha 的 PNG 用 IMREAD_UNCHANGED 读出 4 通道
path = '01_Picture/T-shirt.png'                               # 图像路径

img4 = cv2.imread(path, cv2.IMREAD_UNCHANGED)                 # 以原样模式读取图像
print('UNCHANGED shape:', img4.shape)                         # 期望 (H, W, 4)
print('dtype:', img4.dtype)                                   # 打印数据类型

if img4 is not None and img4.shape[-1] == 4:                  # 如果读取成功且通道数为 4
    b, g, r, a = cv2.split(img4)                              # 拆分 B、G、R、A 通道
    print('alpha 通道 shape:', a.shape)                       # 打印 alpha 通道形状
    print('alpha 最小值/最大值:', a.min(), a.max())           # 打印 alpha 最小值和最大值
else:                                                         # 如果通道数不是 4
    print('该文件没有 4 个通道，可能不带 alpha')              # 打印提示信息

**3）IMREAD_GRAYSCALE**

- 只保留亮度，alpha 也丢失。

**4）合成时怎么用 alpha**

下面用一段可运行代码演示：把带 alpha 的前景贴到白色背景上，
前景和背景按 alpha 归一化后按比例混合。

In [ ]:
# A.4 合成演示：把带 alpha 的前景叠加到白色背景上
path = '01_Picture/T-shirt.png'                               # 图像路径
fg = cv2.imread(path, cv2.IMREAD_UNCHANGED)                   # 以原样模式读取，得到 BGRA

if fg is None or fg.shape[-1] != 4:                           # 如果读取失败或通道数不是 4
    print('请准备一张带 alpha 的 PNG（4 通道 BGRA）再运行此代码')  # 打印提示信息
else:                                                         # 如果条件满足
    b, g, r, a = cv2.split(fg)                                # 拆分通道
    a = a.astype(np.float32) / 255.0                          # 将 alpha 归一化到 0~1
    a = a[..., None]                                          # 扩展维度为 (H, W, 1)

    fg_rgb = fg[:, :, :3].astype(np.float32)                  # 取出前景 BGR 并转为 float32
    bg_rgb = np.full_like(fg_rgb, 255, dtype=np.float32)      # 构造白色背景

    out = fg_rgb * a + bg_rgb * (1 - a)                       # 按 alpha 合成
    out = out.astype(np.uint8)                                # 转回 uint8

    print('合成结果 shape:', out.shape, ' dtype:', out.dtype) # 打印合成结果形状与类型
    plt.figure(figsize=(4, 4))                                # 创建画布
    plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))          # 转为 RGB 后显示
    plt.title('Foreground composited on white')               # 设置标题
    plt.axis('off')                                           # 关闭坐标轴
    plt.show()                                                # 显示图像

### A.5 alpha 归一化到 0~1 后的叠加公式

叠加公式是这样一条：

```
结果颜色 = 前景颜色 × alpha + 背景颜色 × (1 - alpha)
```

其中 alpha 归一化到 0~1。

举例：

- alpha = 1 → 结果完全等于前景；
- alpha = 0 → 结果完全等于背景；
- alpha = 0.5 → 前景和背景各占一半。

这就是“半透明”的数学含义。

**重要理解：这条公式是“叠加公式”，不是“显示公式”。**

它的前提是 **有两个层**：前景 + 背景。
如果没有背景层，它就退化成：

```
结果颜色 = 前景颜色 × alpha
```

但这时会发现问题：**alpha = 0 时结果全黑，alpha = 0.5 时颜色变暗一半**——
这显然不是我们想看到的“半透明”。

为什么会这样？因为 **单纯乘 alpha 并不是显示透明度的正确做法**，它只是叠加公式在“背景=0”时的特例。

真正的显示，一定存在“背景”：

- 屏幕、画布、窗口，本质上 **永远有一个背景**；
- 你在屏幕上看到的一切，都是 **前景叠加到某个背景上的结果**；
- 所谓“没有图层叠加”，其实只是“背景是某块纯色（比如白色或黑色）”，背景仍然存在。

所以严格说：

- **不存在“没有背景的显示”**；
- 你看到的颜色，永远是 `前景 × alpha + 背景 × (1 - alpha)`。

两种常见“背景约定”：

**白底显示**：

- 背景视为白色 (255, 255, 255)；
- 结果 = 前景 × alpha + 255 × (1 - alpha)；
- alpha=0 → 白，alpha=1 → 前景，alpha=0.5 → 前景和白色各一半。

**黑底显示**：

- 背景视为黑色 (0, 0, 0)；
- 结果 = 前景 × alpha；
- alpha=0 → 黑，alpha=1 → 前景，alpha=0.5 → 前景变暗一半。

**所以“结果颜色 = 前景颜色 × alpha”这个式子，只在“背景是黑色”时成立。**

很多图像处理软件，在预览透明图时默认用 **棋盘格背景**：

- 灰色 + 白色的方格；
- 透明区域会显示成棋盘格；
- 这就是在告诉你：“这里其实是透明的，我只是用棋盘格当背景给你看”。

它本质上仍然是叠加，只是背景是棋盘格。

### A.6 如果没有图层叠加，alpha 怎么影响显示

关键点在于：**只要屏幕上显示图像，就一定有“背景”**。所以“没有图层叠加”这个前提，在显示场景里几乎不成立。

下面分三种情况讲清楚 alpha 到底怎么影响显示。

**场景一：图像真的被单独显示（无任何背景）**

这种情况在物理上不存在，因为：

- 屏幕本身就是一层发光面板；
- 窗口背后还有桌面、其他窗口；
- 打印背后还有纸。

但我们可以做一个理想化假设：**背景是全黑，且没有其他光源**。

在这种情况下：

```
显示颜色 = 前景颜色 × alpha + 黑 × (1 - alpha) = 前景颜色 × alpha
```

- alpha = 1 → 显示前景原色；
- alpha = 0.5 → 前景颜色减半，看起来变暗；
- alpha = 0 → 显示全黑（完全看不见前景）。

所以“没有背景”时，alpha 的作用是 **让前景变暗**，而不是“透出后面的东西”，因为后面什么都没有。

**场景二：图像叠加到某个背景上（现实中一定有）**

这才是最常见的显示：

```
显示颜色 = 前景颜色 × alpha + 背景颜色 × (1 - alpha)
```

- alpha = 1 → 显示前景原色，完全盖住背景；
- alpha = 0.5 → 前景和背景各占一半，颜色混合；
- alpha = 0 → 显示背景原色，前景完全看不见。

**这才是 alpha 的本意**：让前景和背后的东西按比例混合。

例如把一张带透明的 PNG 图标贴到网页上：

- 图标的透明区域（alpha=0）→ 完全露出网页背景；
- 图标的半透明区域（alpha=0.5）→ 网页背景和图标的颜色混合；
- 图标的不透明区域（alpha=1）→ 只显示图标本身。

**场景三：直接显示 4 通道 BGRA 给 OpenCV**

这是很多人踩坑的地方：

```python
img = cv2.imread('logo.png', cv2.IMREAD_UNCHANGED)
cv2.imshow('logo', img)
```

- OpenCV 的 `imshow` **不处理 alpha**，只会取前 3 个通道（BGR）；
- alpha 通道被直接忽略；
- 所以透明区域不会叠加背景，而是把 alpha 当成不存在；
- 显示结果可能是：透明区域显示成前景的原始颜色（而不是透出背景）；
- 如果你想要叠加效果，必须自己手动合成（见 A.4 中的合成代码）。

**场景四：看图器 / 浏览器**

大多数看图器、浏览器会：

- 把透明区域用 **棋盘格背景** 显示；
- 其实就是默认叠了一个灰白格子的背景；
- 让你一眼看出“这里其实是透明的”。

它们在背后做的还是那个公式：

```
显示颜色 = 前景颜色 × alpha + 棋盘格颜色 × (1 - alpha)
```

**归纳：alpha 到底怎么影响显示**

- **没有背景（理想黑底）**：alpha 只让前景变暗，`显示 = 前景 × alpha`；
- **有背景（现实）**：alpha 让前景和背景按比例混合，`显示 = 前景 × alpha + 背景 × (1 - alpha)`；
- **OpenCV imshow 直接显示 BGRA**：alpha 被忽略，显示的是“当作 BGR”的前景颜色；
- **看图器 / 浏览器**：默认叠棋盘格背景，等效于 `前景 × alpha + 棋盘格 × (1 - alpha)`。

**一句话总结**

> alpha 在显示时的作用，本质是 **控制前景和背景的混合比例**。
> 如果没有背景，alpha 就只是“让前景变暗”；
> 但现实中的显示一定有背景，所以 alpha 真正的效果是“前景透出背景”。
> `cv2.imshow` 直接显示 4 通道时不会自动合成，alpha 会被忽略。

### A.7 “只有一张图”时，显示背景从哪来

很多人的疑问是：“我手里只有一张图片，哪来的背景？”
这个问题要分两个层面回答。

**一、你手里确实只有一张图片**

从文件角度讲：

- 你有一个 `logo.png` 文件；
- 它里面存的是 4 通道 BGRA（B、G、R、A）；
- 这个文件 **只有这一层**，没有额外的“背景图”。

这部分你说得没错，**你只有一张图**。

**二、但“显示”这个动作，一定会产生背景**

关键在于“显示”和“文件”是两回事：

- **文件里**：只有前景那一层数据；
- **显示时**：图像必须被放到某个“载体”上。

这个“载体”就是所谓的背景。它不一定是另一张图，可能是：

- 屏幕本身（黑屏、白屏、灰底）；
- 软件窗口的背景（比如 OpenCV 窗口默认灰底、浏览器默认白底）；
- 查看器的棋盘格；
- 打印时的白纸。

也就是说：

> 背景不是“另一张图”，而是“图像被放上去的那块底板”。

**三、举个具体例子**

假设你的 PNG 是一个红色半透明圆，alpha=0.5，背景透明区域 alpha=0。

文件里的数据：

- 圆内像素：`BGR=(0,0,255)`，`A=128`；
- 圆外像素：`BGR=(?,?,?)`，`A=0`。

在黑色窗口里显示：

- 圆的显示颜色 = `红 × 0.5 + 黑 × 0.5 = 暗红`；
- 透明区域 = `? × 0 + 黑 × 1 = 黑`（背景露出）。

在白色窗口里显示：

- 圆的显示颜色 = `红 × 0.5 + 白 × 0.5 = 粉红`；
- 透明区域 = `? × 0 + 白 × 1 = 白`。

**同一份文件，显示出来不一样，就是因为“底板”不同。**

**四、如果软件忽略 alpha**

比如 OpenCV 的 `imshow` 直接显示 BGRA：

- 它不读取 alpha；
- 把 `BGR` 三个通道直接画出来；
- 透明区域会显示成“BGR 原始值”，而不是露出背景。

这时候你看到的是“文件里的颜色直接铺出来”，而不是“叠加后的效果”。

**五、那“只有一张图”时，背景是什么**

就是 **显示这块图像的那块区域的颜色**，可能是：

- 你终端窗口的黑色；
- 你浏览器页面的白色；
- 你图像查看器的默认灰底或棋盘格；
- 你打印机的白纸。

它不来自文件，而来自显示环境。

**六、一句话总结**

> “只有一张图”说的是 **文件里只有一层数据**；
> “显示时一定有背景”说的是 **显示这个动作必然把图像放到某个底板上**。
> 这两件事不矛盾。
> alpha 的作用就是决定“前景颜色和底板颜色各占多少”。

## 附录 B：常见图像格式的特点、设计动机与 OpenCV 读取支持

前面 1.3.0 节列出了六种常见格式的 **magic number**。
本附录按 **正文中出现顺序**，依次说明这些格式：

1. **JPEG** — Joint Photographic Experts Group（联合图像专家组）
2. **PNG** — Portable Network Graphics（可移植网络图形）
3. **GIF** — Graphics Interchange Format（图形交换格式）
4. **BMP** — Bitmap（位图）
5. **TIFF** — Tagged Image File Format（标签图像文件格式）
6. **WebP** — Web Picture（网页图片格式，Google 命名）

对每一种格式，我们都回答三个问题：

- 它有什么特点？
- 它为什么被推出？主要解决什么问题？
- `cv2.imread()` 能不能读取它？需要注意什么？

### D.1 JPEG



**特点**

- **有损压缩**：解码后不能完全恢复原始像素，但视觉上通常难以察觉；
- **色彩丰富**：支持 24bit 真彩色（8bit/通道），适合自然图像；
- **文件小**：同等视觉质量下，体积远小于 BMP、TIFF；
- **不支持透明**：没有 alpha 通道；
- **块状伪影**：在高压缩比下，8×8 DCT 块边界会出现“马赛克”。

**为什么推出**

在 JPEG 出现之前，照片类图像常用无压缩的 BMP 或 TIFF 存储，体积巨大，
在有限带宽、有限存储的年代几乎无法传播。
JPEG 用 **DCT + 量化 + 熵编码** 把照片体积压到原来的 1/10 甚至更小，
同时保持肉眼可接受的画质，因此成为 **数码相机、互联网图片** 的通用格式。

**OpenCV 读取支持**

- `cv2.imread()` **默认支持 JPEG**；
- `IMREAD_COLOR`：返回 3 通道 BGR；
- `IMREAD_GRAYSCALE`：返回单通道灰度；
- `IMREAD_UNCHANGED`：JPEG 没有 alpha，所以仍返回 3 通道；
- 真灰度 JPEG（分量数=1）用 `IMREAD_UNCHANGED` 读，返回 `(H, W)` 单通道。

### D.2 PNG



**特点**

- **无损压缩**：解码后与原始像素完全一致；
- **支持透明**：可带 8bit/16bit alpha 通道，形成 RGBA/BGRA；
- **文件相对较大**：同等图像内容下通常大于 JPEG；
- **适合图形、文字、截图**：边缘锐利、色彩块明显，压缩效率高；
- **不适合照片**：自然图像压缩效率不如 JPEG。

**为什么推出**

GIF 受专利限制，且只支持 256 色，无法满足网页对 **真彩色 + 透明** 的需求。
PNG 作为免费、开放、无损、支持 alpha 的替代方案被推出，
专门服务 **网页图形、Logo、截图、需要精确像素的场合**。

**OpenCV 读取支持**

- `cv2.imread()` **默认支持 PNG**；
- `IMREAD_COLOR`：丢弃 alpha，返回 3 通道 BGR；
- `IMREAD_GRAYSCALE`：返回单通道灰度，alpha 丢失；
- `IMREAD_UNCHANGED`：保留 alpha，返回 4 通道 BGRA；
- 带 alpha 的 PNG 上，`IMREAD_GRAYSCALE` 与 `IMREAD_COLOR + cvtColor` 可能不一致，
  详见 **附录 B**。

### D.3 GIF



**特点**

- **无损压缩**（LZW），但 **色彩有限**：最多 256 色；
- **支持动画**：一个文件可包含多帧；
- **支持 1bit 透明**：只有“透明/不透明”两种状态，没有半透明；
- **文件小**：适合低色彩图形与简单动画。

**为什么推出**

GIF 由 CompuServe 在 1987 年推出，用于 **早期网络** 中传输图像。
当时带宽极低，GIF 通过 **调色板 + LZW 压缩** 把低色彩图形压到极小，
并通过多帧机制实现 **简单动画**，成为早期网页的“动图”标准。

**OpenCV 读取支持**

- `cv2.imread()` **通常支持 GIF**，但只读取 **第一帧**；
- 需要读取多帧动画时，应改用 `cv2.VideoCapture` 或专门的 GIF 库（如 `imageio`）；
- 读取后返回 3 通道 BGR（`IMREAD_COLOR`）或单通道（`IMREAD_GRAYSCALE`）；
- GIF 的 1bit 透明在 OpenCV 中通常被忽略或转成固定颜色。

### D.4 BMP



**特点**

- **无压缩**（也可有 RLE 压缩，但少见）：像素数据直接存储；
- **图像质量最好**：解码后与原始像素完全一致；
- **文件体积巨大**：一张 1920×1080 的 24bit BMP 约 6MB；
- **不支持透明**（早期版本），部分扩展支持 alpha；
- **结构简单**：头部字段少，适合作为教学示例。

**为什么推出**

BMP 是 **Windows 系统** 的原生位图格式，用于系统内部、剪贴板、
以及不需要压缩的简单图像存储。
它结构简单、读取快，但代价是 **体积大**，不适合网络传输。

**OpenCV 读取支持**

- `cv2.imread()` **默认支持 BMP**；
- `IMREAD_COLOR`：返回 3 通道 BGR；
- `IMREAD_GRAYSCALE`：返回单通道灰度；
- `IMREAD_UNCHANGED`：若 BMP 带 alpha，则返回 4 通道 BGRA，否则返回 3 通道。

### D.5 TIFF



**特点**

- **无损压缩**（也可有损）：支持 LZW、Deflate、JPEG 等多种压缩方式；
- **支持高位深**：8/12/16/32bit 均可，适合专业图像；
- **支持多页**：一个文件可包含多张图像；
- **支持 alpha**：可带透明通道；
- **文件极大**：因为保留完整细节，通常远大于 JPEG、PNG。

**为什么推出**

TIFF 由 Aldus（后并入 Adobe）在 1986 年推出，面向 **专业印刷、图像编辑、医学影像**。
这些场景要求 **无损、高位深、可扩展**，TIFF 通过“标签 + 灵活字段”
实现了极高的可扩展性，成为专业领域的通用容器格式。

**OpenCV 读取支持**

- `cv2.imread()` **默认支持 TIFF**；
- `IMREAD_COLOR`：返回 3 通道 BGR；
- `IMREAD_GRAYSCALE`：返回单通道灰度；
- `IMREAD_UNCHANGED`：保留原始位深与 alpha，可能返回 uint16、float32 等；
- 多页 TIFF 只读取 **第一页**，需要多页请改用 `tifffile` 等库。

### D.6 WebP



**特点**

- **有损 / 无损皆可**：同一格式支持两种压缩方式；
- **支持透明**：可带 alpha 通道；
- **支持动画**：可替代 GIF；
- **体积小**：同等质量下通常小于 JPEG、PNG、GIF；
- **较新**：2010 年由 Google 推出，生态逐步完善。

**为什么推出**

网页上的图片格式长期由 JPEG、PNG、GIF 三分天下，各有短板：
JPEG 不支持透明，PNG 体积大，GIF 色彩少。
WebP 的目标是 **一个格式通吃三种场景**，用更先进的压缩算法
在同等质量下把体积压到更小，从而 **加速网页加载**。

**OpenCV 读取支持**

- 官方文档把 WebP 列为 **支持格式** 之一；
- 但 WebP 编解码器 **并非在所有环境下都默认编译**：
  - Windows、macOS 的官方预编译包通常包含；
  - Linux 或自行编译的版本可能未开启；
- 若 `cv2.imread()` 返回 `None`，且路径、文件都正确，
  应怀疑当前 OpenCV 构建 **未启用 WebP 支持**；
- 需要稳定读取 WebP 时，可改用 `Pillow` 或 `imageio` 等库，再转换为 numpy 数组。

### D.7 对照表



| 中文名 | 英文全称 | 压缩 | 透明 | 动画 | 位深 | 主要用途 | `cv2.imread()` 默认支持 |
|--------|----------|------|------|------|------|----------|--------------------------|
| JPEG | Joint Photographic Experts Group | 有损 | 否 | 否 | 8bit | 照片、网络图片 | ✅ |
| PNG | Portable Network Graphics | 无损 | 是 | 否 | 8/16bit | 图形、Logo、截图 | ✅ |
| GIF | Graphics Interchange Format | 无损 | 1bit | 是 | 8bit 调色板 | 简单动画、低色彩图形 | ✅（仅第一帧） |
| BMP | Bitmap | 无压缩 | 少见 | 否 | 8/24/32bit | Windows 内部、简单图像 | ✅ |
| TIFF | Tagged Image File Format | 无损/有损 | 是 | 多页 | 8/12/16/32bit | 专业印刷、医学影像 | ✅（仅第一页） |
| WebP | Web Picture | 有损/无损 | 是 | 是 | 8bit | 网页图片、替代 JPG/PNG/GIF | ⚠️ 取决于编译 |

### D.8 一句话总结



> 图像格式的演进，本质是 **在“体积、画质、功能”三者之间做取舍**：
> - **JPEG**（Joint Photographic Experts Group）牺牲无损换体积，服务照片；
> - **PNG**（Portable Network Graphics）牺牲体积换无损 + 透明，服务图形；
> - **GIF**（Graphics Interchange Format）牺牲色彩换动画，服务早期网络；
> - **BMP**（Bitmap）牺牲体积换简单，服务 Windows 系统；
> - **TIFF**（Tagged Image File Format）牺牲体积换专业，服务印刷与医学；
> - **WebP**（Web Picture）试图三者兼顾，服务现代网页。
>
> `cv2.imread()` **默认支持 JPEG、PNG、GIF、BMP、TIFF**；
> **WebP 的支持取决于 OpenCV 编译配置**，遇到读取失败时先怀疑这一点。